# AutoScientist Challenge Part 2 — Data Visualization

**Train a fine-tuned model for the Data Visualization track.**

This notebook:
1. Loads the augmented dataset (5k rows with reasoning traces) from Hugging Face
2. Formats it for SFT (chat messages with enhanced completions)
3. Fine-tunes a Qwen2.5-1.5B-Instruct model with LoRA
4. Evaluates against the 2k validation set using the deterministic eval harness
5. Publishes the fine-tuned weights to Hugging Face

**Runtime:** Attach an L4 (24GB) or A100 GPU. Training takes ~15-30 min.

**Competition:** AutoScientist Challenge Part 2 (Data Visualization)
**Deadline:** August 10, 2026

## 1. Setup

Install dependencies that aren't in the default Clusy image.

In [ ]:
!pip install peft trl -q

In [ ]:
import json
import os
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Attach a GPU runtime (L4 or A100) and re-run.")

os.environ.setdefault("HF_TOKEN", "")  # Set in Clusy Settings → Integrations
HF_USER = "Papajams"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./fine-tuned"
MERGED_DIR = "./fine-tuned-merged"
print(f"\nBase model: {BASE_MODEL}")
print(f"Output: {OUTPUT_DIR}")

## 2. Load and prepare data

Load the augmented dataset from Hugging Face. The augmented rows have:
- `instruction` — task prompt
- `input` — context (data table, CSV, code)
- `output` — original deterministic ground truth
- `enhanced_completion` — augmented output with reasoning traces (we train on this)
- `task` — task type tag

We also load the validation set for evaluation.

In [ ]:
# Load augmented training data from HF
aug_ds = load_dataset(f"{HF_USER}/orbura-dataviz-augmented", split="train")
print(f"Augmented train: {len(aug_ds)} rows")
print(f"Fields: {aug_ds.column_names}")

# Load validation set from HF (original, not augmented)
val_ds = load_dataset(f"{HF_USER}/orbura-dataviz-dataset", split="validation")
print(f"Validation: {len(val_ds)} rows")
print(f"Val fields: {val_ds.column_names}")

# Task distribution
from collections import Counter
task_dist = Counter(aug_ds['task'])
print(f"\nTask distribution (train):")
for t, c in task_dist.most_common():
    print(f"  {t:20s} {c:5d} ({100*c/len(aug_ds):.1f}%)")

In [ ]:
# Format into chat messages for SFT
# We train on the enhanced_completion (with reasoning traces), not the original output
def format_for_sft(example):
    """Build a chat-formatted example using the enhanced completion."""
    user_content = f"{example['instruction']}\n\n{example['input']}"
    assistant_content = example.get('enhanced_completion') or example['output']
    
    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    }

train_formatted = aug_ds.map(format_for_sft, remove_columns=aug_ds.column_names)
print(f"Formatted {len(train_formatted)} training examples")
print(f"\nSample:")
sample = train_formatted[0]
print(f"User (first 200 chars): {sample['messages'][0]['content'][:200]}")
print(f"Assistant (first 200 chars): {sample['messages'][1]['content'][:200]}")
print(f"Assistant length: {len(sample['messages'][1]['content'])} chars")

In [ ]:
# Also prepare a baseline eval set from the validation data
# Save val as JSONL for the eval harness
val_jsonl = "./val_2k.jsonl"
with open(val_jsonl, "w") as f:
    for row in val_ds:
        # Reconstruct the messages format the eval harness expects
        example = {
            "instruction": row["instruction"],
            "input": row["input"],
            "output": row["output"],
            "task": row["task"],
            "messages": [
                {"role": "user", "content": f"{row['instruction']}\n\n{row['input']}"},
                {"role": "assistant", "content": row["output"]},
            ],
        }
        f.write(json.dumps(example) + "\n")
print(f"Saved {len(val_ds)} val examples to {val_jsonl}")

## 3. Load model and tokenizer

Load Qwen2.5-1.5B-Instruct in fp16 for LoRA fine-tuning.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
)
model.config.use_cache = False  # Needed for gradient checkpointing

print(f"Model: {BASE_MODEL}")
print(f"Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max length: {model.config.max_position_embeddings}")

## 4. LoRA configuration

Same recipe that worked for Part 1: LoRA r=32, alpha=64, all linear layers.
This is deliberately modest — the augmented dataset is 5k rows, and we want
the model to learn the reasoning patterns, not memorize the data.

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Tokenize and train

Use TRL's SFTTrainer for supervised fine-tuning. The training set is 5k rows
with enhanced completions (avg 855 chars, 6.2x longer than originals).

In [ ]:
MAX_SEQ_LEN = 1024  # Most enhanced completions fit in 1024 tokens

def format_messages(example):
    """Convert messages list to a single text string for SFT."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_tokenized = train_formatted.map(
    format_messages,
    remove_columns=train_formatted.column_names,
)
print(f"Tokenized {len(train_tokenized)} examples")
print(f"\nSample text (first 300 chars):")
print(train_tokenized[0]["text"][:300])

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_seq_length=MAX_SEQ_LEN,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=3,
    eval_strategy="no",  # We eval after training with our own harness
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tokenized,
    processing_class=tokenizer,
)

print("Starting training...")
print(f"  Epochs: {sft_config.num_train_epochs}")
print(f"  Batch size: {sft_config.per_device_train_batch_size} x {sft_config.gradient_accumulation_steps} = {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  Max seq len: {MAX_SEQ_LEN}")
print(f"  Train examples: {len(train_tokenized)}")
total_steps = (len(train_tokenized) // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)) * sft_config.num_train_epochs
print(f"  Estimated steps: {total_steps}")

In [ ]:
trainer.train()

In [ ]:
# Save the LoRA adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

## 6. Merge LoRA and save full model

Merge the LoRA adapter into the base model so the final weights are
self-contained (no separate adapter needed for inference).

In [ ]:
# Merge LoRA into base model
merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")
print(f"Files:")
for f in os.listdir(MERGED_DIR):
    size = os.path.getsize(os.path.join(MERGED_DIR, f)) / 1e6
    print(f"  {f:40s} {size:.1f} MB")

## 7. Evaluate

Run inference on the 2k validation set and score with the deterministic eval harness.
This is the same harness used for the naive baseline (17.6%).

In [ ]:
# Load the fine-tuned model for inference
del model, trainer, merged_model
torch.cuda.empty_cache()

ft_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
ft_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_model.eval()
print("Fine-tuned model loaded for evaluation.")

In [ ]:
# Scoring functions (from eval_finetuned.py)
import re
import statistics

def parse_data_table(text):
    pairs = re.findall(r"-\s*(\S+):\s*(\d+)", text)
    return {k: int(v) for k, v in pairs} if pairs else None

def extract_code(text):
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    idx = text.find("import matplotlib")
    return text[idx:].strip() if idx >= 0 else text.strip()

def normalize_code(code):
    code = re.sub(r"'(\w+)':", r'"\1":', code)
    code = re.sub(r"\s+", " ", code)
    return code.strip()

def score_chart_qa(pred, ref, input_text):
    pred, ref = pred.strip(), ref.strip()
    if pred == ref:
        return True
    pred_num = re.findall(r"[-+]?\d*\.?\d+", pred)
    ref_num = re.findall(r"[-+]?\d*\.?\d+", ref)
    if pred_num and ref_num:
        try:
            return abs(float(pred_num[0]) - float(ref_num[0])) < 0.15
        except ValueError:
            pass
    if ref.lower() in pred.lower():
        return True
    return False

def score_code(pred, ref):
    pred_code = extract_code(pred)
    if not pred_code or "matplotlib" not in pred_code and "plt" not in pred_code:
        return False
    chart_fns = {"plt.plot", "plt.bar", "plt.scatter", "plt.pie"}
    ref_fn = next((fn for fn in chart_fns if fn in ref), None)
    if ref_fn and ref_fn not in pred_code:
        return False
    ref_values = set(re.findall(r"\d+", ref))
    pred_values = set(re.findall(r"\d+", pred_code))
    if ref_values:
        return len(ref_values & pred_values) / len(ref_values) >= 0.8
    return normalize_code(pred_code) == normalize_code(ref)

def score_chart_choice(pred, ref):
    pred_type = pred.strip().lower().split()[0] if pred.strip() else ""
    ref_type = ref.strip().lower().split()[0] if ref.strip() else ""
    return pred_type == ref_type

def score_code_to_desc(pred, ref):
    pred_lower, ref_lower = pred.lower(), ref.lower()
    chart_types = ["line", "bar", "scatter", "pie"]
    ref_type = next((ct for ct in chart_types if ct in ref_lower), None)
    if ref_type and ref_type not in pred_lower:
        return False
    ref_nums = set(re.findall(r"\d+\.?\d*", ref))
    pred_nums = set(re.findall(r"\d+\.?\d*", pred))
    return len(ref_nums & pred_nums) >= min(2, len(ref_nums))

def score_style_transfer(pred, ref):
    pred_code = extract_code(pred)
    return bool(pred_code) and normalize_code(pred_code) == normalize_code(extract_code(ref))

def score_fix_code(pred, ref):
    pred_code = extract_code(pred)
    return bool(pred_code) and normalize_code(pred_code) == normalize_code(extract_code(ref))

def score_example(example, prediction):
    task = example.get("task", "")
    ref = example.get("output", "").strip()
    pred = prediction.strip()
    if task == "chart_qa":
        return score_chart_qa(pred, ref, example.get("input", ""))
    elif task in ("chart_to_code", "data_to_code"):
        return score_code(pred, ref)
    elif task == "code_to_desc":
        return score_code_to_desc(pred, ref)
    elif task == "style_transfer":
        return score_style_transfer(pred, ref)
    elif task == "fix_code":
        return score_fix_code(pred, ref)
    elif task == "chart_choice":
        return score_chart_choice(pred, ref)
    return pred == ref

print("Scoring functions loaded.")

In [ ]:
# Run inference on the validation set
# Use a subset for speed during development; full 2k for final eval
EVAL_SAMPLES = 2000  # Full val set. Reduce to 200 for quick iteration.

with open(val_jsonl) as f:
    val_examples = [json.loads(line) for line in f if line.strip()]
val_examples = val_examples[:EVAL_SAMPLES]
print(f"Evaluating on {len(val_examples)} examples...")

results = {"total": 0, "correct": 0, "tasks": {}}

for i, ex in enumerate(val_examples):
    messages = [m for m in ex.get("messages", []) if m["role"] == "user"]
    if not messages:
        messages = [{"role": "user", "content": f"{ex['instruction']}\n\n{ex['input']}"}]
    
    # Generate
    text = ft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = ft_tokenizer(text, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        output = ft_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,
            pad_token_id=ft_tokenizer.eos_token_id,
        )
    pred = ft_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    ok = score_example(ex, pred)
    task = ex.get("task", "unknown")
    results["total"] += 1
    results["correct"] += int(ok)
    results["tasks"].setdefault(task, {"total": 0, "correct": 0})
    results["tasks"][task]["total"] += 1
    results["tasks"][task]["correct"] += int(ok)
    
    if (i + 1) % 50 == 0:
        pct = 100 * results["correct"] / results["total"]
        print(f"  [{i+1}/{len(val_examples)}] {pct:.1f}%")

print(f"\n{'='*60}")
print(f"  Fine-tuned model results")
print(f"  Overall: {results['correct']}/{results['total']} ({100*results['correct']/results['total']:.1f}%)")
print(f"{'='*60}")
for task, st in sorted(results["tasks"].items()):
    tpct = 100 * st["correct"] / st["total"] if st["total"] else 0
    print(f"  {task:20s} {st['correct']:4d}/{st['total']:4d}  ({tpct:5.1f}%)")
print()
print(f"Naive baseline was 17.6%. This model: {100*results['correct']/results['total']:.1f}%")

## 8. Publish to Hugging Face

Push the merged model weights to the HF Model Hub.

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi()
repo_id = f"{HF_USER}/orbura-dataviz-qwen2.5-1.5b"

create_repo(repo_id, repo_type="model", exist_ok=True, token=os.environ.get("HF_TOKEN"))
print(f"Uploading model to {repo_id}...")

api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=repo_id,
    repo_type="model",
    token=os.environ.get("HF_TOKEN"),
)
print(f"Model uploaded: https://huggingface.co/{repo_id}")

In [ ]:
# Write a model card
model_card = f"""# Orbura AutoScientist Data Visualization Model

## Model Details

- **Base model:** {BASE_MODEL}
- **Fine-tuning method:** LoRA (r=32, alpha=64) merged into base
- **Dataset:** [Papajams/orbura-dataviz-augmented](https://huggingface.co/datasets/Papajams/orbura-dataviz-augmented) (5,000 rows with reasoning traces)
- **Task:** Data visualization — matplotlib code generation, chart QA, code-to-description, style transfer, bug repair
- **Competition:** AutoScientist Challenge Part 2 — Data Visualization

## Training

- **Train examples:** 5,000 (augmented with reasoning traces via Adaption Adaptive Data)
- **Validation examples:** 2,000 (deterministic, held-out)
- **Epochs:** 3
- **Learning rate:** 2e-4 (cosine schedule)
- **Batch size:** 4 x 4 (effective 16)
- **Max seq length:** 1024
- **LoRA target modules:** q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- **GPU:** {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unknown'}

## Evaluation

| Metric | Naive baseline | Fine-tuned |
|---|---|---|
| Overall | 17.6% | {100*results['correct']/results['total']:.1f}% |

Per-task breakdown:

| Task | Correct/Total | Accuracy |
|---|---|---|
"""
for task, st in sorted(results["tasks"].items()):
    model_card += f"| {task} | {st['correct']}/{st['total']} | {100*st['correct']/st['total']:.1f}% |\n"

model_card += f"""
## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{repo_id}")
tokenizer = AutoTokenizer.from_pretrained("{repo_id}")
```

## License

Apache 2.0 (base model license applies). Dataset and model generated for the AutoScientist Challenge.
"""

with open(f"{MERGED_DIR}/README.md", "w") as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj=f"{MERGED_DIR}/README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="model",
    token=os.environ.get("HF_TOKEN"),
)
print(f"Model card uploaded.")
print(f"\nFinal eval: {100*results['correct']/results['total']:.1f}% (naive baseline: 17.6%)")

## 9. Summary

Copy these results into `MODEL_CARD.md` and `SUBMISSION.md`.

**Next steps:**
1. If the eval is good (>40% overall), publish to Kaggle too
2. Fork this notebook and try Qwen2.5-3B or Mistral-7B for comparison
3. Post on LinkedIn and X using the templates in `social_posts.md`
4. Submit at adaptionlabs.ai before August 10

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Base model:  {BASE_MODEL}")
print(f"Train data:  5,000 augmented rows (reasoning traces)")
print(f"Val data:    {EVAL_SAMPLES} examples")
print(f"Overall:     {100*results['correct']/results['total']:.1f}% ({results['correct']}/{results['total']})")
print(f"Baseline:    17.6%")
print(f"Improvement: +{100*results['correct']/results['total'] - 17.6:.1f}pp")
print(f"\nModel:       https://huggingface.co/{repo_id}")
print(f"Dataset:     https://huggingface.co/datasets/Papajams/orbura-dataviz-augmented")
print(f"\nNext: publish to Kaggle, update MODEL_CARD.md, post on social, submit.")